# MCTS Eval Bracket (Colab)

Evaluate multiple epoch checkpoints with MCTS 400 sims.
Run on G4 for fast GPU inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

!pip install -q numpy numba scipy

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === EDIT THESE ===
CHECKPOINTS = [
    ('2R-ep3', f'{DRIVE}/pillar2r_epoch_3.pt'),
    ('2S-ep3', f'{DRIVE}/pillar2s_epoch_3.pt'),
    ('2S-ep6', f'{DRIVE}/pillar2s_epoch_6.pt'),
]
SIMS = 400
SEEDS = '0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49'
WORKERS = 24
BATCH_SIZE = 128
# ==================

for label, path in CHECKPOINTS:
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'OK: {label} ({os.path.getsize(path)/1e6:.0f} MB)')

In [ ]:
# Run eval bracket — each checkpoint sequentially
for label, path in CHECKPOINTS:
    print(f'\n{"="*60}')
    print(f'=== {label}: {path.split("/")[-1]} ===')
    print(f'{"="*60}\n', flush=True)
    !python -m alphatrain.scripts.eval_parallel \
        --model {path} \
        --device cuda --workers {WORKERS} --simulations {SIMS} \
        --games-per-seed 1 --deterministic --batch-size {BATCH_SIZE} \
        --seeds {SEEDS}